# Chapter 5: Linear Models and Regularization

**Buku:** scikit-learn Cookbook (O'Reilly)
**Topik:** Model linear dan regularisasi (OLS, Ridge, Lasso, ElasticNet) serta perluasan non-linear

## Ringkasan Chapter

Model linear menyatakan target sebagai jumlah berbobot dari fitur:

$$y = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p + \epsilon$$

**Ordinary Least Squares (OLS)** mencari koefisien yang meminimalkan jumlah kuadrat residual. OLS bekerja baik bila fitur sedikit dan tidak saling berkorelasi, tetapi **rapuh** ketika fitur banyak atau berkorelasi kuat (*multikolinearitas*). Solusinya adalah **regularisasi**: menambahkan penalti pada besar koefisien.

| Model | Penalti | Efek |
|:--|:--|:--|
| **OLS** | Tidak ada | Koefisien bisa meledak saat ada multikolinearitas |
| **Ridge** | $\ell_2$: $\alpha\sum\beta_j^2$ | Mengecilkan semua koefisien, tidak pernah tepat nol |
| **Lasso** | $\ell_1$: $\alpha\sum\lvert\beta_j\rvert$ | Membuat sebagian koefisien tepat nol (seleksi fitur) |
| **ElasticNet** | Campuran $\ell_1$ dan $\ell_2$ | Kompromi keduanya |

Bagian akhir membahas **regresi polinomial** dan **spline** agar model linear bisa menangkap pola non-linear.

> Cara pakai: jalankan sel dari atas ke bawah. Setiap sel kode diikuti penjelasan teori dan cara membaca hasilnya.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import warnings
warnings.filterwarnings("ignore")

from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import (LinearRegression, Ridge, Lasso, ElasticNet,
                                  RidgeCV, LassoCV, ElasticNetCV, lasso_path)
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, SplineTransformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

np.random.seed(2024)
pd.set_option("display.width", 120)

print("scikit-learn:", sklearn.__version__)
print("NumPy       :", np.__version__)
print("pandas      :", pd.__version__)

**Penjelasan:** sel ini memuat library dan model yang akan dipakai, lalu mengunci biji acak agar hasil bisa direproduksi.

## 1. OLS: Dasar Model Linear

OLS meminimalkan jumlah kuadrat residual (*Residual Sum of Squares*):

$$\min_{\boldsymbol{\beta}} \ \sum_{i=1}^{n}\left(y_i - \mathbf{x}_i^T\boldsymbol{\beta}\right)^2 \;=\; \min_{\boldsymbol{\beta}} \ \lVert \mathbf{y} - \mathbf{X}\boldsymbol{\beta}\rVert_2^2$$

Solusi tertutupnya (*normal equations*):

$$\hat{\boldsymbol{\beta}} = (\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$$

Kita buktikan bahwa rumus ini sama dengan hasil `LinearRegression` pada contoh kecil.

In [ ]:
rng = np.random.RandomState(0)
X_small = rng.normal(size=(50, 3))
true_beta = np.array([3.0, -2.0, 0.5])
y_small = 1.5 + X_small @ true_beta + rng.normal(0, 0.5, 50)

# Normal equations (tambahkan kolom 1 untuk intercept)
X_b = np.column_stack([np.ones(len(X_small)), X_small])
beta_manual = np.linalg.solve(X_b.T @ X_b, X_b.T @ y_small)

lr = LinearRegression().fit(X_small, y_small)

print("Koefisien sebenarnya (intercept, b1, b2, b3): [1.5, 3.0, -2.0, 0.5]")
print("Normal equations                            :", beta_manual.round(4))
print("LinearRegression                            :", np.r_[lr.intercept_, lr.coef_].round(4))

**Penjelasan hasil:** hasil normal equations dan `LinearRegression` identik, serta dekat dengan koefisien sebenarnya (selisih kecil berasal dari noise). Jadi `LinearRegression` pada dasarnya menyelesaikan persamaan di atas (secara numerik lebih stabil memakai dekomposisi lain, tetapi hasilnya setara).

**Masalah muncul** bila $\mathbf{X}^T\mathbf{X}$ hampir singular, yaitu ketika ada fitur yang hampir merupakan kombinasi linear fitur lain (multikolinearitas). Invers matriks menjadi tidak stabil dan koefisien meledak. Kita lihat langsung di bagian berikut.

## 2. Dataset dengan Multikolinearitas

Kita buat dataset sintetis: 200 sampel, 100 fitur, **hanya 10 fitur pertama yang benar-benar berpengaruh**. Lalu fitur 50 sampai 99 kita ganti menjadi salinan hampir identik dari fitur 0 sampai 49 (ditambah noise kecil). Karena kita mengetahui koefisien sebenarnya, kita dapat memeriksa apakah tiap model menemukan jawaban yang benar.

In [ ]:
rng = np.random.RandomState(123)
X, y, true_coef = make_regression(
    n_samples=200, n_features=100, n_informative=10,
    noise=20, coef=True, shuffle=False, random_state=123
)

# Multikolinearitas: fitur 50-99 = salinan fitur 0-49 + noise kecil
for i in range(50, 100):
    X[:, i] = X[:, i - 50] + rng.normal(0, 0.1, size=200)

print("Ukuran X:", X.shape)
print("Fitur dengan koefisien sebenarnya != 0:", np.nonzero(true_coef)[0])
print("Koefisien sebenarnya (10 pertama):", true_coef[:10].round(1))
print(f"Std target: {y.std():.1f}")
print(f"\nKorelasi feature_0 dengan feature_50: {np.corrcoef(X[:, 0], X[:, 50])[0, 1]:.4f}")
print(f"Korelasi feature_0 dengan feature_1 : {np.corrcoef(X[:, 0], X[:, 1])[0, 1]:.4f}")

eig = np.linalg.eigvalsh(X.T @ X)
print(f"\nEigenvalue terkecil X^T X: {eig.min():.4f} | terbesar: {eig.max():.1f}")
print(f"Condition number         : {eig.max() / eig.min():.2e}")

**Penjelasan hasil:** hanya 10 fitur pertama yang bermakna. Korelasi `feature_0` dengan `feature_50` hampir 1 (salinan), sedangkan dengan `feature_1` mendekati 0. *Condition number* yang sangat besar (rasio eigenvalue terbesar terhadap terkecil) menandakan $\mathbf{X}^T\mathbf{X}$ hampir singular, yaitu kondisi ketika OLS tidak stabil.

Kita bagi data 70/30. Sampel latih (140) hanya sedikit lebih banyak dari jumlah fitur (100), sehingga OLS rawan overfitting.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=123)
print("Latih:", X_train.shape, "| Uji:", X_test.shape)

def evaluate(name, model, store):
    model.fit(X_train, y_train)
    est = model[-1] if hasattr(model, "steps") else model
    coef = est.coef_
    p_te, p_tr = model.predict(X_test), model.predict(X_train)
    store.append({
        "Model": name,
        "R2 latih": r2_score(y_train, p_tr),
        "R2 uji": r2_score(y_test, p_te),
        "RMSE uji": np.sqrt(mean_squared_error(y_test, p_te)),
        "Norma L2 koef": np.linalg.norm(coef),
        "Koef != 0": int((np.abs(coef) > 1e-8).sum()),
    })
    return model

rows = []
ols = evaluate("OLS", make_pipeline(StandardScaler(), LinearRegression()), rows)
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Penjelasan hasil:** R² pada data latih sangat tinggi (hampir 1) sedangkan pada data uji lebih rendah. Selisih ini menandakan **overfitting**: dengan 100 koefisien dan hanya 140 sampel, OLS menyesuaikan diri dengan noise. Norma L2 koefisien yang besar juga mengisyaratkan koefisien yang tidak stabil. Fitur sudah distandarisasi (`StandardScaler`) supaya koefisien antar fitur sebanding, hal yang nanti penting untuk regularisasi.

### 2.1 Bukti ketidakstabilan koefisien OLS

Jika OLS stabil, melatihnya ulang pada sampel data yang sedikit berbeda harus menghasilkan koefisien yang mirip. Kita uji dengan *bootstrap*.

In [ ]:
rng = np.random.RandomState(0)
coefs_ols, coefs_ridge = [], []
for _ in range(100):
    idx = rng.choice(len(X_train), len(X_train), replace=True)
    coefs_ols.append(make_pipeline(StandardScaler(), LinearRegression()).fit(X_train[idx], y_train[idx])[-1].coef_)
    coefs_ridge.append(make_pipeline(StandardScaler(), Ridge(alpha=10)).fit(X_train[idx], y_train[idx])[-1].coef_)
coefs_ols, coefs_ridge = np.array(coefs_ols), np.array(coefs_ridge)

print("Ketidakstabilan koefisien antar 100 bootstrap")
print(f"{'':<22}{'OLS':>12}{'Ridge(a=10)':>14}")
for j in [0, 50]:
    print(f"Std feature_{j:<9}{coefs_ols[:, j].std():>12.2f}{coefs_ridge[:, j].std():>14.2f}")
print(f"Rata-rata std semua  {coefs_ols.std(axis=0).mean():>12.2f}{coefs_ridge.std(axis=0).mean():>14.2f}")

# Hubungan antara koef feature_0 dan feature_50 (salinan satu sama lain)
print("\nKorelasi koefisien feature_0 vs feature_50 antar bootstrap:")
print(f"  OLS  : {np.corrcoef(coefs_ols[:, 0], coefs_ols[:, 50])[0, 1]:.3f}")
print(f"  Ridge: {np.corrcoef(coefs_ridge[:, 0], coefs_ridge[:, 50])[0, 1]:.3f}")

**Penjelasan hasil:** koefisien OLS berubah-ubah jauh lebih besar antar bootstrap dibanding Ridge (std lebih tinggi). Korelasi koefisien `feature_0` dan `feature_50` pada OLS sangat negatif: ketika satu naik, pasangannya turun. Itulah efek "tarik-menarik": karena kedua fitur hampir sama, OLS tidak bisa menentukan bagaimana membagi bobot di antara keduanya, sehingga memberi bobot besar berlawanan tanda yang saling meniadakan. Prediksi masih masuk akal, tetapi **koefisien tidak dapat ditafsirkan**. Ridge menstabilkan hal ini.

## 3. Ridge Regression ($\ell_2$)

Ridge menambahkan penalti kuadrat koefisien:

$$\mathcal{L}_{Ridge} = \sum_{i}(y_i - \hat{y}_i)^2 + \alpha\sum_{j}\beta_j^2$$

Solusi tertutupnya hanya menambahkan $\alpha\mathbf{I}$ pada diagonal:

$$\hat{\boldsymbol{\beta}}_{Ridge} = (\mathbf{X}^T\mathbf{X} + \alpha\mathbf{I})^{-1}\mathbf{X}^T\mathbf{y}$$

Penambahan $\alpha\mathbf{I}$ membuat matriks selalu dapat dibalik dan stabil. Kita verifikasi manual (pada data yang sudah distandarisasi dan target yang sudah dipusatkan).

In [ ]:
sc = StandardScaler().fit(X_train)
Xs = sc.transform(X_train)
yc = y_train - y_train.mean()

alpha = 10.0
beta_manual = np.linalg.solve(Xs.T @ Xs + alpha * np.eye(Xs.shape[1]), Xs.T @ yc)
ridge_sk = Ridge(alpha=alpha).fit(Xs, y_train)

print("Selisih maksimum koefisien manual vs Ridge:", np.abs(beta_manual - ridge_sk.coef_).max())
print("Norma L2 OLS-terstandarisasi :", np.linalg.norm(ols[-1].coef_).round(2))
print("Norma L2 Ridge (alpha=10)    :", np.linalg.norm(ridge_sk.coef_).round(2))

**Penjelasan hasil:** selisihnya mendekati nol, jadi rumus tertutup di atas memang yang dihitung `Ridge`. Norma koefisien Ridge lebih kecil daripada OLS: itulah efek *shrinkage*. Ridge mengecilkan koefisien menuju nol tetapi **tidak pernah membuatnya tepat nol**.

Sekarang kita latih Ridge, Lasso, dan ElasticNet.

## 4. Lasso ($\ell_1$) dan ElasticNet

**Lasso** memakai penalti nilai mutlak:

$$\mathcal{L}_{Lasso} = \frac{1}{2n}\sum_i (y_i - \hat{y}_i)^2 + \alpha\sum_j \lvert\beta_j\rvert$$

Bentuk penalti $\ell_1$ memiliki "sudut" di nol, sehingga banyak koefisien jatuh tepat ke 0. Lasso melakukan **seleksi fitur otomatis**. (Catatan: scikit-learn menulis suku kesalahan Lasso dengan faktor $\frac{1}{2n}$, sehingga skala $\alpha$ Lasso tidak langsung sebanding dengan $\alpha$ Ridge.)

**ElasticNet** menggabungkan keduanya, dikontrol parameter `l1_ratio` (1 = Lasso murni, 0 = Ridge murni):

$$\mathcal{L}_{EN} = \frac{1}{2n}\sum_i (y_i - \hat{y}_i)^2 + \alpha\left(\rho\sum_j\lvert\beta_j\rvert + \frac{1-\rho}{2}\sum_j\beta_j^2\right)$$

Intuisi grup fitur berkorelasi: Lasso cenderung memilih satu wakil dan membuang sisanya (pilihannya bisa acak), Ridge mempertahankan semuanya tetapi tidak memilih, sedangkan ElasticNet cenderung memilih kelompok bersama-sama.

In [ ]:
rows = []
ols   = evaluate("OLS", make_pipeline(StandardScaler(), LinearRegression()), rows)
ridge = evaluate("Ridge (alpha=10)", make_pipeline(StandardScaler(), Ridge(alpha=10)), rows)
lasso = evaluate("Lasso (alpha=1)", make_pipeline(StandardScaler(), Lasso(alpha=1.0, max_iter=50000)), rows)
enet  = evaluate("ElasticNet (alpha=1, l1=0.5)", make_pipeline(StandardScaler(), ElasticNet(alpha=1.0, l1_ratio=0.5, max_iter=50000)), rows)

print(pd.DataFrame(rows).round(3).to_string(index=False))

**Penjelasan hasil:** bandingkan kolom demi kolom.

- **R² uji:** Ridge dan Lasso memperbaiki OLS; selisih R² latih dan uji mengecil (overfitting berkurang).
- **Norma L2 koefisien:** semua model teregularisasi punya norma lebih kecil dari OLS.
- **Koef != 0:** OLS dan Ridge mempertahankan semua 100 koefisien, sedangkan Lasso hanya mempertahankan sebagian (seleksi fitur) dan ElasticNet berada di antara keduanya.

Perhatikan bahwa ElasticNet dengan `alpha=1, l1_ratio=0.5` di sini kurang baik, karena $\alpha$ tersebut terlalu besar untuk data ini. Ini mengingatkan bahwa **$\alpha$ harus dituning**, bukan ditebak (bagian 6).

### 4.1 Apakah Lasso menemukan fitur yang benar?

Kita tahu fitur 0-9 adalah fitur sebenarnya. Kita cek fitur mana yang dipilih Lasso.

In [ ]:
lasso_coef = lasso[-1].coef_
selected = np.nonzero(np.abs(lasso_coef) > 1e-8)[0]

true_set = set(range(10))
copy_set = set(range(50, 60))      # salinan dari 0-9
sel_set = set(selected)

print("Jumlah fitur terpilih Lasso:", len(selected))
print("  Fitur sebenarnya (0-9) yang terpilih :", sorted(sel_set & true_set))
print("  Salinannya (50-59) yang terpilih     :", sorted(sel_set & copy_set))
print("  Fitur noise lain yang terpilih       :", sorted(sel_set - true_set - copy_set))

**Penjelasan hasil:** Lasso biasanya memilih fitur-fitur sebenarnya, tetapi karena fitur 50-59 adalah salinan dari 0-9, ia kadang memilih salinannya sebagai pengganti, atau memilih keduanya. Inilah kelemahan Lasso pada fitur berkorelasi: pilihannya di antara pasangan hampir identik bersifat arbitrer. Jumlah total fitur terpilih jauh lebih sedikit daripada 100, sehingga model jauh lebih sederhana.

### 4.2 Membandingkan koefisien tiap model

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 8), sharex=True)
for ax, (name, model) in zip(axes.flat, [("OLS", ols), ("Ridge (a=10)", ridge), ("Lasso (a=1)", lasso), ("ElasticNet (a=1, l1=0.5)", enet)]):
    c = model[-1].coef_
    ax.bar(range(100), c, width=1.0, color="steelblue")
    ax.set_title(f"{name} | norma L2 = {np.linalg.norm(c):.0f} | non-zero = {(np.abs(c) > 1e-8).sum()}")
    ax.set_ylabel("Koefisien (terstandarisasi)")
for ax in axes[1]:
    ax.set_xlabel("Indeks fitur")
plt.tight_layout()
plt.show()

**Penjelasan hasil:** pada panel OLS, koefisien tersebar lebar pada seluruh 100 fitur, termasuk fitur noise. Pada Ridge, koefisien mengecil dan tersebar lebih merata antara fitur asli dan salinannya. Pada Lasso, sebagian besar batang hilang (tepat nol) dan tersisa beberapa koefisien besar, sebagian besar pada fitur 0-9 atau salinan 50-59. ElasticNet berada di antara keduanya.

## 5. Teori Regularisasi: Bias-Variance dan Pengaruh $\alpha$

$\alpha$ mengatur trade-off bias-variance:

- $\alpha$ kecil: bias rendah, varians tinggi (mendekati OLS, rawan overfitting).
- $\alpha$ besar: bias tinggi, varians rendah (koefisien terlalu ditekan, underfitting).

Titik terbaik ada di tengah. Kita pindai nilai $\alpha$ dan amati error latih serta uji.

In [ ]:
alphas = np.logspace(-3, 3, 40)
res = {"Ridge": [], "Lasso": []}
for a in alphas:
    for name, M in [("Ridge", Ridge), ("Lasso", Lasso)]:
        m = make_pipeline(StandardScaler(), M(alpha=a, max_iter=50000) if name == "Lasso" else M(alpha=a)).fit(X_train, y_train)
        res[name].append((mean_squared_error(y_train, m.predict(X_train)),
                          mean_squared_error(y_test, m.predict(X_test))))
res = {k: np.array(v) for k, v in res.items()}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, name in zip(axes, ["Ridge", "Lasso"]):
    ax.plot(np.log10(alphas), res[name][:, 0], label="MSE latih")
    ax.plot(np.log10(alphas), res[name][:, 1], label="MSE uji")
    best = alphas[res[name][:, 1].argmin()]
    ax.axvline(np.log10(best), color="red", ls="--", label=f"alpha terbaik ~ {best:.2f}")
    ax.set_xlabel("log10(alpha)"); ax.set_ylabel("MSE"); ax.set_yscale("log")
    ax.set_title(name); ax.legend()
plt.tight_layout()
plt.show()

for name in res:
    i = res[name][:, 1].argmin()
    print(f"{name:<6}: alpha terbaik (data uji) = {alphas[i]:.3f}, MSE uji = {res[name][i, 1]:.1f}")
print(f"OLS   : MSE uji = {mean_squared_error(y_test, ols.predict(X_test)):.1f}")

**Penjelasan hasil:** MSE **latih** selalu naik seiring $\alpha$ membesar (model makin dikekang). MSE **uji** membentuk kurva berbentuk U: turun dulu karena overfitting berkurang, mencapai titik minimum, lalu naik lagi karena model terlalu ditekan (underfitting). Garis merah menandai $\alpha$ terbaik pada pemindaian ini. Pada $\alpha$ sangat kecil, hasil mendekati OLS.

**Catatan metodologis:** memilih $\alpha$ memakai data uji seperti di atas hanya untuk ilustrasi. Pada praktik sebenarnya, $\alpha$ harus dipilih lewat **cross-validation pada data latih**, supaya data uji tetap murni untuk evaluasi akhir (bagian 6).

### 5.1 Coefficient path

*Coefficient path* menampilkan perubahan setiap koefisien ketika $\alpha$ divariasikan.

In [ ]:
Xs_tr = StandardScaler().fit_transform(X_train)
a_path, coefs_path, _ = lasso_path(Xs_tr, y_train - y_train.mean(), alphas=np.logspace(-2, 1.5, 60))

ridge_path = np.array([Ridge(alpha=a * 10).fit(Xs_tr, y_train).coef_ for a in a_path])

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for j in range(100):
    c = "crimson" if j < 10 else ("orange" if 50 <= j < 60 else "lightgray")
    axes[0].plot(np.log10(a_path), coefs_path[j], color=c, lw=1.2 if c != "lightgray" else 0.6)
    axes[1].plot(np.log10(a_path * 10), ridge_path[:, j], color=c, lw=1.2 if c != "lightgray" else 0.6)
axes[0].set_title("Lasso path"); axes[0].set_xlabel("log10(alpha)"); axes[0].set_ylabel("Koefisien")
axes[1].set_title("Ridge path"); axes[1].set_xlabel("log10(alpha)")
axes[0].plot([], [], color="crimson", label="fitur sebenarnya (0-9)")
axes[0].plot([], [], color="orange", label="salinannya (50-59)")
axes[0].plot([], [], color="lightgray", label="lainnya")
axes[0].legend()
plt.tight_layout()
plt.show()

**Penjelasan hasil:** pada Lasso path (kiri), saat $\alpha$ membesar koefisien **jatuh ke nol satu per satu dan tepat nol**; koefisien fitur sebenarnya (merah) bertahan paling lama, sedangkan fitur noise (abu-abu) cepat hilang. Pada Ridge path (kanan), koefisien **mengecil mulus mendekati nol tetapi tidak pernah persis nol**. Inilah perbedaan visual utama $\ell_1$ dan $\ell_2$.

## 6. Memilih $\alpha$ dengan Cross-Validation

scikit-learn menyediakan `RidgeCV`, `LassoCV`, dan `ElasticNetCV` yang memilih $\alpha$ (dan `l1_ratio`) lewat cross-validation, hanya memakai data latih.

In [ ]:
ridge_cv = make_pipeline(StandardScaler(), RidgeCV(alphas=np.logspace(-2, 3, 60))).fit(X_train, y_train)
lasso_cv = make_pipeline(StandardScaler(), LassoCV(cv=5, max_iter=50000, random_state=0)).fit(X_train, y_train)
enet_cv  = make_pipeline(StandardScaler(), ElasticNetCV(l1_ratio=[0.1, 0.5, 0.7, 0.9, 0.95], cv=5, max_iter=50000, random_state=0)).fit(X_train, y_train)

print("Hasil pemilihan lewat CV (hanya pakai data latih):")
print(f"  RidgeCV      : alpha = {ridge_cv[-1].alpha_:.3f}")
print(f"  LassoCV      : alpha = {lasso_cv[-1].alpha_:.3f}")
print(f"  ElasticNetCV : alpha = {enet_cv[-1].alpha_:.3f}, l1_ratio = {enet_cv[-1].l1_ratio_}")

rows = []
for name, m in [("OLS", ols), ("RidgeCV", ridge_cv), ("LassoCV", lasso_cv), ("ElasticNetCV", enet_cv)]:
    c = m[-1].coef_
    rows.append({"Model": name,
                 "R2 uji": r2_score(y_test, m.predict(X_test)),
                 "RMSE uji": np.sqrt(mean_squared_error(y_test, m.predict(X_test))),
                 "Koef != 0": int((np.abs(c) > 1e-8).sum()),
                 "Norma L2": np.linalg.norm(c)})
print()
print(pd.DataFrame(rows).round(3).to_string(index=False))

**Penjelasan hasil:** dengan $\alpha$ yang dipilih lewat CV, semua model teregularisasi mengalahkan OLS pada data uji, dan ElasticNetCV/LassoCV serta RidgeCV memberi hasil yang bersaing. Data uji hanya dipakai di tahap akhir ini. Pada dataset ini perbedaan kecil antar ketiganya tidak boleh dibaca sebagai pemenang pasti karena data uji hanya 60 sampel; yang jelas konsisten adalah semuanya lebih baik daripada OLS.

Perhatikan juga kolom `Koef != 0`: Ridge mempertahankan seluruh 100 fitur, sedangkan Lasso dan ElasticNet menghasilkan model yang lebih ringkas (lebih mudah ditafsirkan).

### 6.1 Pentingnya scaling sebelum regularisasi

Penalti memperlakukan semua koefisien sama besar, padahal besar koefisien bergantung pada skala fitur. Fitur berskala kecil butuh koefisien besar sehingga terkena penalti lebih berat. Kita buktikan dengan mengubah skala satu fitur yang penting.

In [ ]:
X_tr_mod, X_te_mod = X_train.copy(), X_test.copy()
X_tr_mod[:, 0] *= 0.01        # perkecil skala feature_0 (fitur informatif)
X_te_mod[:, 0] *= 0.01

for scaled in [False, True]:
    steps = [StandardScaler()] if scaled else []
    m = make_pipeline(*steps, Ridge(alpha=10)).fit(X_tr_mod, y_train)
    print(f"Ridge {'DENGAN' if scaled else 'TANPA '} scaling | R2 uji = {m.score(X_te_mod, y_test):.4f} "
          f"| koef feature_0 = {m[-1].coef_[0]:.2f}")
m0 = make_pipeline(Ridge(alpha=10)).fit(X_train, y_train)
print(f"Ridge tanpa scaling, skala asli | R2 uji = {m0.score(X_test, y_test):.4f}")

**Penjelasan hasil:** ketika `feature_0` diperkecil skalanya 100 kali tanpa standarisasi, Ridge menekannya habis-habisan karena koefisiennya harus sangat besar untuk memberi kontribusi yang sama, dan R² uji turun dibanding skala asli. Dengan `StandardScaler`, hasilnya tidak terpengaruh oleh perubahan skala. **Kesimpulan:** selalu standarisasi fitur sebelum Ridge, Lasso, atau ElasticNet, dan letakkan di dalam `Pipeline`.

## 7. Perluasan Non-Linear: Polinomial dan Spline

Model linear bisa menangkap pola non-linear bila kita **mengubah fitur**: model tetap linear terhadap *parameter*, tetapi non-linear terhadap $x$. Kita pakai data dengan tren linear ditambah gelombang sinus:

$$y = 2x + 27\sin(x/8) + \epsilon, \qquad \epsilon \sim \mathcal{N}(0, 4^2)$$

In [ ]:
rng = np.random.RandomState(123)
x_all = rng.uniform(-50, 50, 600)
y_all = 2 * x_all + 27 * np.sin(x_all / 8) + rng.normal(0, 4, 600)
X_all = x_all.reshape(-1, 1)

# Latih pada 80 titik, sisanya sebagai data uji
Xn_tr, Xn_te, yn_tr, yn_te = train_test_split(X_all, y_all, train_size=80, random_state=123)

plt.figure(figsize=(9, 4))
plt.scatter(Xn_tr, yn_tr, s=15, label="latih (80)")
plt.scatter(Xn_te, yn_te, s=6, alpha=0.3, label="uji (520)")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.title("Data non-linear")
plt.show()

lin = LinearRegression().fit(Xn_tr, yn_tr)
print(f"Regresi linear biasa: R2 uji = {lin.score(Xn_te, yn_te):.4f}")

**Penjelasan hasil:** data memperlihatkan tren naik dengan gelombang di atasnya. Garis lurus hanya menangkap tren $2x$ dan tetap memberi R² yang cukup tinggi, karena tren linearnya dominan, tetapi gelombangnya terlewat. Residualnya akan berpola, bukan acak.

### 7.1 Regresi Polinomial

Fitur $x$ diperluas menjadi $\{x, x^2, \dots, x^d\}$, lalu dipasang regresi linear:

$$y = \beta_0 + \beta_1 x + \beta_2 x^2 + \cdots + \beta_d x^d$$

Derajat $d$ mengontrol fleksibilitas. Karena pangkat tinggi menghasilkan angka sangat besar, kita **standarisasi** fitur polinomial agar stabil secara numerik.

In [ ]:
def poly_model(d, reg=None):
    return make_pipeline(
        StandardScaler(),                               # skalakan x dulu (rentang -50..50 -> ~N(0,1))
        PolynomialFeatures(degree=d, include_bias=False),
        StandardScaler(),
        reg if reg is not None else LinearRegression(),
    )

degrees = [1, 2, 3, 5, 7, 9, 12, 15]
rows = []
for d in degrees:
    m = poly_model(d).fit(Xn_tr, yn_tr)
    rows.append({"Derajat": d,
                 "R2 latih": m.score(Xn_tr, yn_tr),
                 "R2 uji": m.score(Xn_te, yn_te),
                 "RMSE uji": np.sqrt(mean_squared_error(yn_te, m.predict(Xn_te)))})
poly_df = pd.DataFrame(rows)
print(poly_df.round(4).to_string(index=False))

**Penjelasan hasil:** R² latih naik terus ketika derajat bertambah (model makin fleksibel selalu makin cocok dengan data latih). R² **uji** naik dulu lalu mencapai puncak di derajat menengah, dan setelah itu bisa stagnan atau turun karena model mulai mengikuti noise (*overfitting*), terutama di tepi rentang $x$. Derajat 1-3 terlalu kaku (*underfitting*) untuk menangkap gelombang. Inilah pola klasik bias-variance: pilih derajat berdasarkan **performa data uji/validasi**, bukan data latih.

In [ ]:
xg = np.linspace(-50, 50, 600).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
axes[0].scatter(Xn_tr, yn_tr, s=15, c="gray", label="latih")
for d, c in [(1, "blue"), (5, "green"), (9, "red"), (15, "purple")]:
    m = poly_model(d).fit(Xn_tr, yn_tr)
    axes[0].plot(xg, m.predict(xg), color=c, label=f"derajat {d}")
axes[0].plot(xg, 2 * xg + 27 * np.sin(xg / 8), "k--", alpha=0.5, label="fungsi asli")
axes[0].set_ylim(-150, 150); axes[0].legend(); axes[0].set_title("Polinomial berbagai derajat")

axes[1].plot(poly_df["Derajat"], poly_df["R2 latih"], "o-", label="R2 latih")
axes[1].plot(poly_df["Derajat"], poly_df["R2 uji"], "s-", label="R2 uji")
axes[1].set_xlabel("Derajat polinomial"); axes[1].set_ylabel("R2"); axes[1].legend()
axes[1].set_title("Underfitting vs overfitting")
plt.tight_layout()
plt.show()

**Penjelasan hasil:** panel kiri: derajat 1 (biru) lurus dan meleset dari gelombang; derajat menengah mengikuti gelombang dengan baik; derajat sangat tinggi (ungu) bisa meliuk-liuk berlebihan dan menyimpang tajam di tepi (*Runge phenomenon*). Panel kanan merangkum jarak antara R² latih dan uji. Garis putus-putus hitam adalah fungsi sebenarnya.

### 7.2 Regularisasi menyelamatkan polinomial berderajat tinggi

Polinomial derajat tinggi yang overfit bisa diperbaiki dengan Ridge, yaitu **regularisasi bertemu fitur non-linear**.

In [ ]:
d = 15
plain = poly_model(d).fit(Xn_tr, yn_tr)
rows = [{"Model": f"Derajat {d} tanpa regularisasi", "R2 uji": plain.score(Xn_te, yn_te)}]
for a in [0.01, 0.1, 1, 10, 100]:
    m = poly_model(d, Ridge(alpha=a)).fit(Xn_tr, yn_tr)
    rows.append({"Model": f"Derajat {d} + Ridge(alpha={a})", "R2 uji": m.score(Xn_te, yn_te)})
print(pd.DataFrame(rows).round(4).to_string(index=False))

best_a = RidgeCV(alphas=np.logspace(-3, 3, 30))
m_best = poly_model(d, best_a).fit(Xn_tr, yn_tr)
print(f"\nRidgeCV memilih alpha = {m_best[-1].alpha_:.3f}, R2 uji = {m_best.score(Xn_te, yn_te):.4f}")

**Penjelasan hasil:** derajat 15 tanpa regularisasi memiliki R² uji lebih rendah daripada versi yang diberi penalti Ridge dengan $\alpha$ yang tepat. Regularisasi menekan koefisien suku berpangkat tinggi sehingga kurva tidak meliuk berlebihan. Jadi alternatif memilih derajat dengan hati-hati adalah memakai derajat tinggi yang diregularisasi dan membiarkan $\alpha$ mengatur kompleksitas.

### 7.3 Spline

**Spline** memasang polinomial berderajat rendah *per potongan* (di antara titik-titik **knot**) dengan syarat mulus di tiap knot. Lebih stabil daripada polinomial global karena pengaruh tiap bagian bersifat lokal. `SplineTransformer` membuat basis B-spline yang lalu dikombinasikan oleh regresi linear.

In [ ]:
rows = []
for k in [3, 5, 7, 10, 15, 25]:
    m = make_pipeline(SplineTransformer(n_knots=k, degree=3), LinearRegression()).fit(Xn_tr, yn_tr)
    rows.append({"n_knots": k, "R2 latih": m.score(Xn_tr, yn_tr), "R2 uji": m.score(Xn_te, yn_te)})
spl_df = pd.DataFrame(rows)
print(spl_df.round(4).to_string(index=False))

plt.figure(figsize=(10, 5))
plt.scatter(Xn_tr, yn_tr, s=15, c="gray", label="latih")
for k, c in [(3, "blue"), (7, "green"), (25, "red")]:
    m = make_pipeline(SplineTransformer(n_knots=k, degree=3), LinearRegression()).fit(Xn_tr, yn_tr)
    plt.plot(xg, m.predict(xg), color=c, label=f"spline, knots={k}")
plt.plot(xg, 2 * xg + 27 * np.sin(xg / 8), "k--", alpha=0.5, label="fungsi asli")
plt.ylim(-150, 150); plt.legend(); plt.title("Spline dengan jumlah knot berbeda")
plt.show()

**Penjelasan hasil:** dengan sedikit knot (3), spline terlalu kaku dan melewatkan gelombang. Dengan jumlah knot menengah (sekitar 7-10), kurva mengikuti gelombang dengan baik dan R² uji tinggi. Dengan terlalu banyak knot (25) pada hanya 80 titik latih, model mulai mengikuti noise dan R² uji dapat turun. Semua R² di sini dihitung pada **data uji**, bukan pada data latih (menghitung pada data latih akan selalu menunjukkan bahwa makin banyak knot makin bagus, yang menyesatkan).

Spline juga bisa diregularisasi: kombinasikan dengan `Ridge` dan pilih `n_knots` serta $\alpha$ lewat `GridSearchCV`.

In [ ]:
from sklearn.model_selection import GridSearchCV

pipe = make_pipeline(SplineTransformer(degree=3), Ridge())
grid = GridSearchCV(
    pipe,
    {"splinetransformer__n_knots": [5, 7, 10, 15, 20],
     "ridge__alpha": [0.001, 0.01, 0.1, 1, 10]},
    cv=KFold(5, shuffle=True, random_state=0),
)
grid.fit(Xn_tr, yn_tr)

print("Parameter terbaik:", grid.best_params_)
print(f"Skor CV (R2)     : {grid.best_score_:.4f}")
print(f"R2 pada data uji : {grid.score(Xn_te, yn_te):.4f}")

**Penjelasan hasil:** `GridSearchCV` memilih jumlah knot dan kekuatan regularisasi berdasarkan cross-validation pada data latih, lalu skor akhir dihitung pada data uji yang belum pernah dilihat. Nama parameter pada pipeline dibentuk dari nama langkah (huruf kecil) diikuti dua garis bawah, misalnya `splinetransformer__n_knots`.

## 8. Tambahan: Regularisasi pada Klasifikasi (Regresi Logistik)

Regularisasi juga dipakai pada klasifikasi. Pada `LogisticRegression`, parameter **`C` adalah kebalikan dari kekuatan regularisasi**: $C$ kecil = regularisasi kuat, $C$ besar = regularisasi lemah. Kita bandingkan penalti $\ell_1$ dan $\ell_2$ pada dataset kanker payudara (30 fitur).

In [ ]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression

Xc, yc = load_breast_cancer(return_X_y=True)
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(Xc, yc, test_size=0.3, random_state=0, stratify=yc)

rows = []
for C in [0.001, 0.01, 0.1, 1, 10, 100]:
    for pen in ["l1", "l2"]:
        m = make_pipeline(StandardScaler(), LogisticRegression(penalty=pen, C=C, solver="liblinear", max_iter=5000)).fit(Xc_tr, yc_tr)
        coef = m[-1].coef_.ravel()
        rows.append({"C": C, "Penalti": pen, "Akurasi uji": m.score(Xc_te, yc_te),
                     "Koef != 0": int((np.abs(coef) > 1e-8).sum())})
tab = pd.DataFrame(rows).pivot(index="C", columns="Penalti", values=["Akurasi uji", "Koef != 0"])
print(tab.round(4).to_string())

**Penjelasan hasil:** pada $C$ sangat kecil (regularisasi kuat), penalti $\ell_1$ menghasilkan sedikit koefisien non-nol (bahkan nol semuanya bila terlalu kuat), sedangkan $\ell_2$ tetap menyisakan 30 koefisien. Seiring $C$ membesar, jumlah koefisien non-nol pada $\ell_1$ naik. Akurasi biasanya sudah baik pada $C$ menengah, sehingga model $\ell_1$ bisa mencapai akurasi serupa dengan lebih sedikit fitur. Pola ini identik dengan Lasso dan Ridge pada regresi.

## Ringkasan Akhir Chapter 5

| Konsep | Poin penting |
|:--|:--|
| **OLS** | $\hat{\beta} = (X^TX)^{-1}X^Ty$. Cepat dan mudah ditafsirkan, tetapi tidak stabil saat fitur banyak atau berkorelasi (overfitting, koefisien tidak dapat ditafsirkan) |
| **Ridge ($\ell_2$)** | Mengecilkan semua koefisien, menstabilkan multikolinearitas, tidak pernah menghasilkan nol |
| **Lasso ($\ell_1$)** | Menghasilkan koefisien nol (seleksi fitur), tetapi pilihan di antara fitur berkorelasi bisa arbitrer |
| **ElasticNet** | Gabungan $\ell_1$ dan $\ell_2$ (`l1_ratio`), baik untuk kelompok fitur berkorelasi |
| **Memilih $\alpha$** | Lewat cross-validation (`RidgeCV`, `LassoCV`, `ElasticNetCV`, `GridSearchCV`), bukan data uji |
| **Scaling** | Wajib sebelum regularisasi karena penalti bergantung pada skala fitur |
| **Polinomial dan spline** | Memperluas model linear ke pola non-linear; kendalikan kompleksitas dengan derajat atau knot dan regularisasi |
| **Regresi logistik** | Parameter `C` = kebalikan kekuatan regularisasi; `l1` memberi model yang jarang (sparse) |

**Tiga pelajaran utama:**

1. **Regularisasi mengorbankan sedikit bias untuk menurunkan varians besar**, terutama saat fitur banyak atau berkorelasi.
2. **Standarisasi dan Pipeline** wajib; pilih $\alpha$ dengan cross-validation pada data latih.
3. **Evaluasi pada data uji, bukan data latih**, terutama saat membandingkan derajat polinomial atau jumlah knot.